# XiangQi-AI：Google Colab CUDA 训练

本 Notebook 采用 Drive 常驻源码方案。源码、临时文件、依赖缓存、日志、Replay、checkpoint 和最终模型全部保存在 `MyDrive/XiangQi-AI/`。请在 Colab 的“运行时 → 更改运行时类型”中选择 NVIDIA GPU，然后从上到下运行。

训练默认每局最多 **512 个完整回合（1024 ply）**；如果此前没有自然终局，到达上限按和棋生成标签。所有走子都来自现有规则引擎的合法走法；若出现将死，对局立即结束并按真实胜负训练。

> 同一个运行目录一次只能启动一个训练进程。正式训练前建议先运行 GPU smoke。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 1. 创建全部持久化目录

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/XiangQi-AI")
SOURCE_DIR = DRIVE_ROOT / "source"
RUNS_DIR = DRIVE_ROOT / "runs"
TEMP_DIR = DRIVE_ROOT / "temp"
CACHE_DIR = DRIVE_ROOT / "cache"
LOGS_DIR = DRIVE_ROOT / "logs"

for directory in (
    DRIVE_ROOT,
    RUNS_DIR,
    TEMP_DIR,
    CACHE_DIR / "pip",
    CACHE_DIR / "torch",
    LOGS_DIR,
):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TEMP_DIR)
os.environ["PIP_CACHE_DIR"] = str(CACHE_DIR / "pip")
os.environ["TORCH_HOME"] = str(CACHE_DIR / "torch")
print(f"持久化根目录：{DRIVE_ROOT}")

## 2. 首次克隆或安全更新 Drive 中的源码

In [ ]:
import subprocess

REPOSITORY_URL = "https://github.com/HongLouWang/XiangQi-AI.git"
if not (SOURCE_DIR / ".git").is_dir():
    if SOURCE_DIR.exists() and any(SOURCE_DIR.iterdir()):
        raise RuntimeError(f"源码目录非空但不是 Git 仓库：{SOURCE_DIR}")
    subprocess.run(["git", "clone", REPOSITORY_URL, str(SOURCE_DIR)], check=True)

In [ ]:
branch = subprocess.run(
    ["git", "-C", str(SOURCE_DIR), "rev-parse", "--abbrev-ref", "HEAD"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
commit = subprocess.run(
    ["git", "-C", str(SOURCE_DIR), "rev-parse", "HEAD"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()
source_status = subprocess.run(
    ["git", "-C", str(SOURCE_DIR), "status", "--short"],
    check=True,
    text=True,
    capture_output=True,
).stdout
print(f"分支：{branch}\n提交：{commit}\ngit status --short:")
print(source_status or "（干净）")

In [ ]:
current_status = subprocess.run(
    ["git", "-C", str(SOURCE_DIR), "status", "--short"],
    check=True,
    text=True,
    capture_output=True,
).stdout
if current_status.strip():
    print("源码存在本地修改，为保护修改已跳过自动更新。")
    print(current_status)
else:
    subprocess.run(
        ["git", "-C", str(SOURCE_DIR), "pull", "--ff-only"],
        check=True,
    )

## 3. 安装 AI 依赖并强制检查 CUDA

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{SOURCE_DIR}[ai]"],
    check=True,
)

In [ ]:
DEVICE = "cuda:0"
subprocess.run(["nvidia-smi"], check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "如果没有可用的 NVIDIA CUDA GPU，训练将停止；请把 Colab 运行时改为 GPU。"
    )
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
device_index = torch.device(DEVICE).index or 0
free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
print("GPU 数量:", torch.cuda.device_count())
print("当前 GPU:", torch.cuda.get_device_name(device_index))
print(f"显存：空闲 {free_bytes / 2**30:.2f} GiB / 总计 {total_bytes / 2**30:.2f} GiB")

## 4. 编辑正式训练参数

In [ ]:
RUN_NAME = "colab-gpu"
TARGET_GAMES = 10_000
MAX_FULL_MOVES = 512
DEVICE = "cuda:0"
SIMULATIONS = 64
CHANNELS = 64
RESIDUAL_BLOCKS = 4
BATCH_SIZE = 128
CHECKPOINT_INTERVAL_GAMES = 10
GAME_RETRY_LIMIT = 2
SEED = 0
RUN_DIR = RUNS_DIR / RUN_NAME
LOG_PATH = LOGS_DIR / f"{RUN_NAME}.log"
PID_PATH = DRIVE_ROOT / "colab-training.pid"
LOCK_DIR = RUN_DIR / ".colab-training.lock"
LOCK_STARTUP_GRACE_SECONDS = 120
print(f"训练目录：{RUN_DIR}")

## 5. 可选：先做一次真实 CUDA smoke

这会在 Drive 的独立目录训练 1 局、每局最多 1 个完整回合，不会碰正式训练目录。

In [ ]:
SMOKE_RUN_DIR = RUNS_DIR / "colab-gpu-smoke"
SMOKE_FINAL_MODEL = SMOKE_RUN_DIR / "final_model.pt"
SMOKE_CHECKPOINTS = tuple(
    SMOKE_RUN_DIR / name for name in ("checkpoint-a.pt", "checkpoint-b.pt")
)
smoke_complete = SMOKE_FINAL_MODEL.is_file() and any(
    checkpoint.is_file() for checkpoint in SMOKE_CHECKPOINTS
)
smoke_has_data = SMOKE_RUN_DIR.exists() and any(SMOKE_RUN_DIR.iterdir())
if smoke_complete:
    torch.load(SMOKE_FINAL_MODEL, map_location="cpu", weights_only=True)
    for checkpoint in SMOKE_CHECKPOINTS:
        if checkpoint.is_file():
            torch.load(checkpoint, map_location="cpu", weights_only=True)
    print("Smoke 已完成，跳过重复训练。")
elif smoke_has_data:
    raise RuntimeError(
        f"Smoke 目录存在不完整数据：{SMOKE_RUN_DIR}。请人工检查后改用新的目录名。"
    )
else:
    smoke_command = [
        sys.executable,
        "-m",
        "ai",
        "train",
        "--run-dir",
        str(SMOKE_RUN_DIR),
        "--games",
        "1",
        "--full-moves",
        "1",
        "--device",
        DEVICE,
        "--simulations",
        "1",
        "--channels",
        "8",
        "--residual-blocks",
        "1",
        "--batch-size",
        "1",
        "--checkpoint-interval-games",
        "1",
    ]
    subprocess.run(smoke_command, cwd=SOURCE_DIR, env=os.environ.copy(), check=True)
    subprocess.run(
        [sys.executable, "-m", "ai", "status", "--run-dir", str(SMOKE_RUN_DIR)],
        cwd=SOURCE_DIR,
        env=os.environ.copy(),
        check=True,
    )
    assert SMOKE_FINAL_MODEL.is_file()
    assert any(checkpoint.is_file() for checkpoint in SMOKE_CHECKPOINTS)
    torch.load(SMOKE_FINAL_MODEL, map_location="cpu", weights_only=True)
    print("CUDA smoke 已通过。")

## 6. 定义后台训练与控制函数

In [ ]:
import json
import math
import time
import uuid


def process_matches(pid: int) -> bool:
    cmdline_path = Path(f"/proc/{pid}/cmdline")
    try:
        raw = cmdline_path.read_bytes()
    except (FileNotFoundError, PermissionError, ProcessLookupError):
        return False
    parts = [part.decode(errors="replace") for part in raw.split(b"\0") if part]
    return "-m" in parts and "ai" in parts and str(RUN_DIR).encode() in raw.split(b"\0")


def _read_lock_metadata() -> dict[str, object]:
    try:
        payload = json.loads((LOCK_DIR / "owner.json").read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError, OSError):
        return {}
    return payload if isinstance(payload, dict) else {}


def _write_lock_metadata(token: str, pid: int | None) -> None:
    payload = {
        "token": token,
        "run_dir": str(RUN_DIR),
        "pid": pid,
        "created_at": time.time(),
    }
    temporary = RUN_DIR / f".colab-lock-owner-{token}.tmp"
    temporary.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")
    os.replace(temporary, LOCK_DIR / "owner.json")


def _remove_owned_lock(token: str) -> None:
    metadata = _read_lock_metadata()
    if metadata.get("token") != token:
        return
    try:
        (LOCK_DIR / "owner.json").unlink()
        LOCK_DIR.rmdir()
    except FileNotFoundError:
        pass


def _remove_just_created_lock(token: str) -> None:
    owner_path = LOCK_DIR / "owner.json"
    metadata = _read_lock_metadata()
    if metadata and metadata.get("token") != token:
        return
    if owner_path.exists():
        if metadata.get("token") != token:
            return
        owner_path.unlink()
    temporary = RUN_DIR / f".colab-lock-owner-{token}.tmp"
    if temporary.exists():
        temporary.unlink()
    try:
        LOCK_DIR.rmdir()
    except FileNotFoundError:
        pass


def _reclaim_stale_lock() -> None:
    stale_dir = RUN_DIR / f".colab-training.lock.stale-{uuid.uuid4().hex}"
    try:
        LOCK_DIR.rename(stale_dir)
    except FileNotFoundError:
        return
    owner_path = stale_dir / "owner.json"
    if owner_path.exists():
        owner_path.unlink()
    try:
        stale_dir.rmdir()
    except OSError as error:
        raise RuntimeError(f"陈旧锁含未知文件，请人工检查：{stale_dir}") from error


def acquire_training_lock() -> str:
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    token = uuid.uuid4().hex
    for _ in range(3):
        try:
            LOCK_DIR.mkdir()
        except FileExistsError:
            metadata = _read_lock_metadata()
            pid = metadata.get("pid")
            if isinstance(pid, int) and process_matches(pid):
                raise RuntimeError(f"该运行目录已有训练进程 PID={pid}")
            try:
                lock_modified_at = LOCK_DIR.stat().st_mtime
            except FileNotFoundError:
                continue
            created_at = metadata.get("created_at")
            if not isinstance(created_at, (int, float)) or not math.isfinite(
                created_at
            ):
                created_at = lock_modified_at
            lock_age = time.time() - created_at
            if pid is None and lock_age < LOCK_STARTUP_GRACE_SECONDS:
                raise RuntimeError("另一个 Colab 会话正在启动该运行目录")
            _reclaim_stale_lock()
            continue
        try:
            _write_lock_metadata(token, pid=None)
        except BaseException:
            _remove_just_created_lock(token)
            raise
        return token
    raise RuntimeError("训练锁持续发生竞争，请稍后重试")


def training_command(resume: bool) -> list[str]:
    if resume:
        return [
            sys.executable,
            "-m",
            "ai",
            "resume",
            "--run-dir",
            str(RUN_DIR),
            "--device",
            DEVICE,
        ]
    return [
        sys.executable,
        "-m",
        "ai",
        "train",
        "--run-dir",
        str(RUN_DIR),
        "--games",
        str(TARGET_GAMES),
        "--full-moves",
        str(MAX_FULL_MOVES),
        "--device",
        DEVICE,
        "--simulations",
        str(SIMULATIONS),
        "--channels",
        str(CHANNELS),
        "--residual-blocks",
        str(RESIDUAL_BLOCKS),
        "--batch-size",
        str(BATCH_SIZE),
        "--checkpoint-interval-games",
        str(CHECKPOINT_INTERVAL_GAMES),
        "--game-retry-limit",
        str(GAME_RETRY_LIMIT),
        "--seed",
        str(SEED),
    ]


def _saved_pid() -> int | None:
    try:
        return int(PID_PATH.read_text(encoding="utf-8").strip())
    except (FileNotFoundError, ValueError):
        return None


def start_background_training(resume: bool = False) -> int:
    if resume and not any(
        (RUN_DIR / name).is_file() for name in ("checkpoint-a.pt", "checkpoint-b.pt")
    ):
        raise RuntimeError(f"找不到可恢复 checkpoint：{RUN_DIR}")
    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
    lock_token = acquire_training_lock()
    process = None
    try:
        with LOG_PATH.open("a", encoding="utf-8") as log_file:
            process = subprocess.Popen(
                training_command(resume=resume),
                cwd=SOURCE_DIR,
                env=os.environ.copy(),
                stdout=log_file,
                stderr=subprocess.STDOUT,
                start_new_session=True,
            )
        _write_lock_metadata(lock_token, pid=process.pid)
        temporary_pid_path = PID_PATH.with_suffix(".tmp")
        temporary_pid_path.write_text(f"{process.pid}\n", encoding="utf-8")
        os.replace(temporary_pid_path, PID_PATH)
    except BaseException:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=30)
        _remove_owned_lock(lock_token)
        raise
    time.sleep(2)
    if process.poll() is not None:
        _remove_owned_lock(lock_token)
        raise RuntimeError(f"训练启动失败，请查看日志：{LOG_PATH}")
    print(f"训练已在后台启动，PID={process.pid}，日志={LOG_PATH}")
    return process.pid


def run_ai_command(
    *arguments: str, capture_output: bool = False
) -> subprocess.CompletedProcess[str]:
    return subprocess.run(
        [sys.executable, "-m", "ai", *arguments],
        cwd=SOURCE_DIR,
        env=os.environ.copy(),
        check=True,
        text=True,
        capture_output=capture_output,
    )

## 7. 启动新的正式训练（只运行一次）

In [ ]:
TRAINING_PID = start_background_training(resume=False)

## 8. 查询状态和最近日志（可反复运行）

In [ ]:
status_result = run_ai_command("status", "--run-dir", str(RUN_DIR), capture_output=True)
print(status_result.stdout)
if LOG_PATH.is_file():
    log_lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(log_lines[-100:]))

## 9. 安全暂停（需要暂停时才运行）

暂停会等待当前棋局原子提交并保存 checkpoint；最长等待 30 分钟。

In [ ]:
run_ai_command("pause", "--run-dir", str(RUN_DIR))
deadline = time.monotonic() + 30 * 60
while True:
    result = run_ai_command("status", "--run-dir", str(RUN_DIR), capture_output=True)
    status_data = json.loads(result.stdout)
    print(status_data)
    if status_data.get("phase") in {"paused", "completed", "failed"}:
        break
    if time.monotonic() >= deadline:
        raise TimeoutError("等待安全暂停超过 30 分钟，请查看日志和 status。")
    time.sleep(10)

## 10. 追加目标局数（需要追加时才运行）

In [ ]:
ADDITIONAL_GAMES = 5_000
extend_result = run_ai_command(
    "extend",
    "--run-dir",
    str(RUN_DIR),
    "--games",
    str(ADDITIONAL_GAMES),
    capture_output=True,
)
print(extend_result.stdout)

## 11. 从 checkpoint 恢复（暂停或重新连接 Colab 后运行）

In [ ]:
resume_preview = training_command(resume=True)
print("恢复命令：", resume_preview)
TRAINING_PID = start_background_training(resume=True)

## 12. 验证持久化文件和安全加载模型

In [ ]:
artifact_paths = [
    RUN_DIR / "checkpoint-a.pt",
    RUN_DIR / "checkpoint-b.pt",
    RUN_DIR / "final_model.pt",
    RUN_DIR / "latest.json",
    RUN_DIR / "status.json",
    RUN_DIR / "replay" / "manifest.json",
    LOG_PATH,
]
for path in artifact_paths:
    print(("存在" if path.exists() else "尚未生成"), path)
    if path.suffix == ".pt" and path.is_file():
        torch.load(path, map_location="cpu", weights_only=True)
print("已有 .pt 文件均已使用安全模式加载。")

## Colab 断线后的恢复顺序

1. 重新选择 GPU 运行时。
2. 依次运行挂载 Drive、目录、源码更新、安装、CUDA 检查和配置单元格。
3. 运行函数定义单元格。
4. 先运行“查询状态和最近日志”。
5. 确认旧进程已不存在后，再运行“从 checkpoint 恢复”。

不要同时启动两个进程操作同一个 `RUN_DIR`。Notebook 不手工修改 Replay 或 checkpoint；恢复兼容性由训练器负责。